## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-05-00-55-15 +0000,nypost,Lockheed Martin paying $29.5M settlement after...,https://nypost.com/2026/11/04/business/lockhee...
1,2026-11-05-00-51-06 +0000,nyt,‘It Was Never Enough’: Susan Collins Couldn’t ...,https://www.nytimes.com/2026/11/04/us/politics...
2,2026-11-05-00-51-04 +0000,nypost,California’s biggest election loser is Dave Re...,https://nypost.com/2026/11/04/us-news/californ...
3,2026-11-05-00-48-15 +0000,nyt,"In a City of Start-Ups, the Golden State Valky...",https://www.nytimes.com/2026/11/04/us/golden-s...
4,2026-11-05-00-46-48 +0000,nypost,Socialist Nithya Raman widens lead over Karen ...,https://nypost.com/2026/11/04/us-news/nithya-r...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2450/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
16,trump,72
260,democrats,53
46,race,46
102,midterm,42
54,senate,38


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
233,2026-11-04-18-06-00 +0000,wsj,"To win control of the House and Senate, Democr...",https://www.wsj.com/politics/elections/the-vot...,272
351,2026-11-04-11-34-05 +0000,chitri,Democrats win the US House and Senate as voter...,https://www.chicagotribune.com/2026/11/04/poll...,264
174,2026-11-04-20-06-00 +0000,wsj,"Democrats won control of Congress, a majority ...",https://www.wsj.com/politics/elections/how-dem...,251
285,2026-11-04-15-51-00 +0000,wsj,The Democrats’ big victories in the House and ...,https://www.wsj.com/politics/elections/trump-p...,244
82,2026-11-04-22-37-00 +0000,wsj,Democrats’ midterm sweep has shaken up the 202...,https://www.wsj.com/politics/elections/democra...,212


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
233,272,2026-11-04-18-06-00 +0000,wsj,"To win control of the House and Senate, Democr...",https://www.wsj.com/politics/elections/the-vot...
439,161,2026-11-04-04-40-50 +0000,chitri,Iowa Senate race: Democrat Josh Turek wins ele...,https://www.chicagotribune.com/2026/11/03/josh...
82,159,2026-11-04-22-37-00 +0000,wsj,Democrats’ midterm sweep has shaken up the 202...,https://www.wsj.com/politics/elections/democra...
88,134,2026-11-04-22-31-00 +0000,wsj,Vice President JD Vance blamed the war in Iran...,https://www.wsj.com/politics/policy/vances-bre...
283,101,2026-11-04-15-55-00 +0000,wsj,Elon Musk’s political fortunes have hit a blue...,https://www.wsj.com/politics/elections/elon-mu...
239,96,2026-11-04-18-01-15 +0000,chitri,Column: Lake County voters don’t buy school sa...,https://www.chicagotribune.com/2026/11/04/colu...
35,95,2026-11-04-23-53-52 +0000,latimes,California Democrats crush GOP opponents in fi...,https://www.latimes.com/california/story/2026-...
449,74,2026-11-04-04-28-16 +0000,latimes,Ugarte takes early lead for South L.A. City Co...,https://www.latimes.com/california/story/2026-...
497,61,2026-11-04-01-04-38 +0000,chitri,Georgia Senate race: Sen. Jon Ossoff wins reel...,https://www.chicagotribune.com/2026/11/03/geor...
503,57,2026-11-04-01-00-07 +0000,chitri,Results come in for Chicago’s school board rac...,https://www.chicagotribune.com/2026/11/03/chic...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
